# TimelyMT Research MVP: Kaggle Git-Clone Runner
Import only this notebook into Kaggle, enable a GPU accelerator and Internet, then run all cells in order. The notebook clones the public repository and stops after frozen TRAIN/DEV selection; it never executes the held-out evaluation split.

In [ ]:
REPO_URL = "https://github.com/MinhCYB/TimelyMT.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/TimelyMT"

INFERENCE_BATCH_SIZE = 3

HF_CACHE_DIR = "/kaggle/temp/huggingface"

## CLONE REPOSITORY
Clone shallow `main` from GitHub. Rerunning this cell refreshes only the disposable Kaggle checkout while retaining valid ignored run artifacts in the current session. No path under `/kaggle/input` is read or modified.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo = Path(REPO_DIR)
expected_remote = REPO_URL.removesuffix(".git").rstrip("/").lower()

def run(command, *, cwd=None, capture_output=False):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(
        list(map(str, command)), cwd=cwd, check=True, text=True,
        capture_output=capture_output,
    )

valid_checkout = False
if repo.exists() and (repo / ".git").is_dir():
    remote = run(["git", "remote", "get-url", "origin"], cwd=repo, capture_output=True).stdout.strip()
    valid_checkout = remote.removesuffix(".git").rstrip("/").lower() == expected_remote

if valid_checkout:
    run(["git", "fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=repo)
    run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=repo)
else:
    if repo.exists():
        print(f"Removing unexpected disposable path: {repo}", flush=True)
        shutil.rmtree(repo)
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, repo])

os.chdir(repo)
commit = run(["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True).stdout.strip()
status = run(["git", "status", "--short"], cwd=repo, capture_output=True).stdout
print(f"Experiment commit: {commit}")
print("git status --short:")
print(status if status else "(clean)")

## ENVIRONMENT SETUP
Use Kaggle's preinstalled CUDA-enabled PyTorch. Install the editable project and its constrained dependencies from `pyproject.toml` without upgrading unrelated packages.
<!-- Compatibility heading: ## MODEL/CACHE SETUP -->

In [ ]:
TRANSFORMERS_REQUIREMENT = "transformers>=4.57.6,<5.0.0"
hf_cache = Path(HF_CACHE_DIR)
hf_cache.mkdir(parents=True, exist_ok=True)
os.environ.update({
    "HF_HOME": HF_CACHE_DIR,
    "HF_HUB_CACHE": str(hf_cache / "hub"),
    "TRANSFORMERS_CACHE": str(hf_cache / "transformers"),
    "TOKENIZERS_PARALLELISM": "false",
})

try:
    import torch
except ImportError as error:
    raise RuntimeError("Kaggle's preinstalled PyTorch is unavailable; do not continue") from error

run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=repo)

import sacrebleu
import sentencepiece
import sklearn
import transformers

transformers_major_minor = tuple(map(int, transformers.__version__.split(".")[:2]))
if not ((4, 57) <= transformers_major_minor < (5, 0)):
    raise RuntimeError(f"Unsupported transformers version: {transformers.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a Kaggle GPU accelerator before continuing.")

print(f"Python: {sys.version.split()[0]}")
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"sentencepiece: {getattr(sentencepiece, '__version__', 'unknown')}")
print(f"sklearn: {sklearn.__version__}")
print(f"sacrebleu: {sacrebleu.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Inference batch size: {INFERENCE_BATCH_SIZE}")

def cli(*args):
    command = [sys.executable, "-u", "-m", "timelymt.research.cli", *args]
    run(command, cwd=repo)

def require_file(relative_path):
    path = repo / relative_path
    if not path.is_file():
        raise RuntimeError(f"Missing required artifact: {path}")
    return path

## PRECHECK
Validate the cloned frozen Dataset v1 and experimental split before downloading EnViT5. This cell never reacquires TED data or rebuilds M0.

In [ ]:
import json
from timelymt.data.canonical.core import load_canonical_talk
from timelymt.data.manifest.core import validate_split_manifest
from timelymt.data.pipeline.qa import stable_checksum, validate_dataset
from timelymt.data.translation_artifacts import runtime_talk_from_canonical, stable_fingerprint
from timelymt.research.cli import DATASET_CHECKSUM, SPLIT_CHECKSUM, _manifests

required_paths = [
    "configs/translator/envit5.json",
    "configs/experiments/research-mvp.json",
    "data/manifests/streaming-dataset.json",
    "data/manifests/timelymt-streaming-dataset-v1.json",
    "data/splits/experimental.json",
    "schemas/streaming-talk.schema.json",
    "src/timelymt/research/cli.py",
]
for required in required_paths:
    require_file(required)

manifest = json.loads(require_file("data/manifests/streaming-dataset.json").read_text(encoding="utf-8"))
snapshot = json.loads(require_file("data/manifests/timelymt-streaming-dataset-v1.json").read_text(encoding="utf-8"))
split = json.loads(require_file("data/splits/experimental.json").read_text(encoding="utf-8"))
expected_ids = {row["talk_id"] for row in manifest["talks"]}
canonical_paths = {path.parent.name: path for path in (repo / "data/streaming/processed").glob("*/streaming-talk.json")}
if len(expected_ids) != 17 or set(canonical_paths) != expected_ids:
    raise RuntimeError({
        "expected_count": len(expected_ids),
        "actual_count": len(canonical_paths),
        "missing": sorted(expected_ids - set(canonical_paths)),
        "extra": sorted(set(canonical_paths) - expected_ids),
    })

dataset_result = validate_dataset(manifest, project_root=repo)
validate_split_manifest(split, manifest)
for row in manifest["talks"]:
    document = load_canonical_talk(repo / row["canonical_path"])
    runtime_talk_from_canonical(
        document, split_manifest=split,
        observed_through_token_index=len(document["stream"]["tokens"]) - 1,
    )
if not (
    dataset_result["manifest_checksum"] == snapshot["manifest_checksum"] == DATASET_CHECKSUM
    and stable_checksum(split) == stable_fingerprint(split) == snapshot["split_manifest_checksum"] == SPLIT_CHECKSUM
):
    raise RuntimeError("Frozen Dataset v1 or split identity changed")
_manifests()
print(json.dumps({
    "dataset": dataset_result,
    "snapshot_version": snapshot["snapshot_version"],
    "split_checksum": stable_fingerprint(split),
    "split_counts": {name: len(ids) for name, ids in split["splits"].items()},
    "canonical_runtime_talks_validated": len(canonical_paths),
}, indent=2, sort_keys=True))

## MODEL/CACHE SETUP
Download the frozen EnViT5 revision into Kaggle temporary storage, never into Git. The experiment does not fine-tune EnViT5.

In [ ]:
MODEL_ID = "VietAI/envit5-translation"
MODEL_REVISION = "840bc88104d5a4277af740eaedb024df8c3093e7"

from huggingface_hub import snapshot_download
from timelymt.translator.envit5 import load_config

translator_config = load_config(repo / "configs/translator/envit5.json")
if not (
    translator_config.frozen
    and translator_config.model_id == MODEL_ID
    and translator_config.model_revision == MODEL_REVISION
):
    raise RuntimeError("Translator config is not the expected frozen EnViT5 revision")
snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, cache_dir=os.environ["HF_HUB_CACHE"])
print(f"Pinned model cached under: {HF_CACHE_DIR}")

## MODEL PREFLIGHT
Run one tiny real translation through the existing TimelyMT translator API. This verifies the pinned revision, CUDA float16 inference, and removal of EnViT5's leading `vi:` control tag.

In [ ]:
from timelymt.translator.envit5 import EnViT5Translator

smoke_translator = EnViT5Translator(translator_config)
smoke_result = smoke_translator.translate("Artificial intelligence helps people")
runtime_info = smoke_translator.runtime_info()
if runtime_info["model_revision"] != MODEL_REVISION:
    raise RuntimeError(f"Loaded unexpected model revision: {runtime_info['model_revision']}")
if runtime_info["device"] != "cuda" or runtime_info["dtype"] != "float16":
    raise RuntimeError(f"Unexpected inference runtime: {runtime_info}")
if smoke_result.translated_text.startswith("vi:"):
    raise RuntimeError("Normalized EnViT5 output still contains the leading vi: control tag")
print(json.dumps({**runtime_info, "normalized_output": smoke_result.translated_text}, indent=2, ensure_ascii=False))
del smoke_translator
torch.cuda.empty_cache()

## FULL TRAIN — TIMELYMT PSEUDO LABELS
Generate full TRAIN future-stability supervision on GPU. A fresh public clone starts at 0/12 talks; rerunning after interruption reuses valid completed talk-level artifacts.

If CUDA OOM occurs, change only `INFERENCE_BATCH_SIZE`: 3 -> 2 -> 1.
<!-- Compatibility heading: ## TIMELYMT TRAIN PSEUDO -->

In [ ]:
cli("pseudo", "--split", "train", "--batch-size", str(INFERENCE_BATCH_SIZE))

## FULL TRAIN — ZHANG 2020 MU SUPERVISION
Generate full TRAIN oracle supervision for the frozen MU literature adaptation on GPU. This is TRAIN-only supervision; MU runtime remains causal.

In [ ]:
cli("mu-supervision", "--split", "train", "--batch-size", str(INFERENCE_BATCH_SIZE))

## FULL DEV — TIMELYMT PSEUDO LABELS
Generate full DEV supervision on GPU for freeze prerequisites and provenance. Valid talk-level artifacts are resumable.
<!-- Compatibility heading: ## TIMELYMT DEV PSEUDO -->

In [ ]:
cli("pseudo", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE))

## FULL DEV — ZHANG 2020 MU SUPERVISION
Generate full DEV MU supervision on GPU for freeze prerequisites.
<!-- Compatibility heading: ## MU TRAIN/DEV SUPERVISION -->

In [ ]:
cli("mu-supervision", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE))

## VALIDATE SUPERVISION
Require full, publishable TRAIN/DEV manifests and exact split coverage before classifier training.
<!-- Compatibility heading: ## VALIDATE -->

In [ ]:
for stage, split_name in (
    ("validate-pseudo", "train"),
    ("validate-pseudo", "dev"),
    ("validate-mu", "train"),
    ("validate-mu", "dev"),
):
    cli(stage, "--split", split_name)
for relative in (
    "data/policy/pseudo_labels/train/manifest.json",
    "data/policy/pseudo_labels/dev/manifest.json",
    "data/policy/mu_zhang2020/train/manifest.json",
    "data/policy/mu_zhang2020/dev/manifest.json",
):
    document = json.loads(require_file(relative).read_text(encoding="utf-8"))
    if document.get("artifact_status") != "full" or document.get("publishable") is not True:
        raise RuntimeError(f"Supervision is not full/publishable: {relative}")

## TRAIN P0
Train the lightweight P0 policy classifier on CPU. EnViT5 remains frozen.
<!-- Compatibility heading: ## TRAIN P0/P1/P2 -->

In [ ]:
cli("train", "--pseudo-labels", "data/policy/pseudo_labels/train/manifest.json", "--variant", "P0")
require_file("checkpoints/policy/P0.joblib")
require_file("checkpoints/policy/P0.metadata.json")

## TRAIN P1
Train the lightweight P1 policy classifier on CPU.

In [ ]:
cli("train", "--pseudo-labels", "data/policy/pseudo_labels/train/manifest.json", "--variant", "P1")
require_file("checkpoints/policy/P1.joblib")
require_file("checkpoints/policy/P1.metadata.json")

## TRAIN P2
Train the lightweight P2 classifier. P2 history remains system-generated target history only.

In [ ]:
cli("train", "--pseudo-labels", "data/policy/pseudo_labels/train/manifest.json", "--variant", "P2")
require_file("checkpoints/policy/P2.joblib")
require_file("checkpoints/policy/P2.metadata.json")

## TRAIN MU
Train the lightweight Zhang-2020 MU adaptation classifier on CPU.

In [ ]:
cli("train-mu", "--pseudo-labels", "data/policy/mu_zhang2020/train/manifest.json")
require_file("checkpoints/policy/mu_zhang2020.joblib")
require_file("checkpoints/policy/mu_zhang2020.metadata.json")

## DEV FIXED BASELINES
Run all frozen Fixed-N and Fixed-Time DEV baselines on GPU. Completed prediction files are resumable.
<!-- Compatibility heading: ## DEV BASELINES -->

In [ ]:
FIXED = [
    "fixed_n_4", "fixed_n_8", "fixed_n_12",
    "fixed_time_1600", "fixed_time_3200", "fixed_time_4800",
]
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *FIXED)

## DEV LOCAL AGREEMENT STYLE
Run TimelyMT's frozen `local_agreement_style` k=2 and k=3 heuristic baselines on GPU.

In [ ]:
STYLE = ["local_agreement_style_k2", "local_agreement_style_k3"]
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *STYLE)

## DEV LOCAL AGREEMENT LA-2
Run the frozen literature LA-2 adaptation on DEV using GPU inference.
<!-- Compatibility heading: ## DEV LA-2 -->

In [ ]:
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", "local_agreement_la2")

## DEV MU
Run the trained Zhang-2020 MU adaptation on DEV using GPU inference.
<!-- Compatibility heading: ## DEV MU ROLLOUT -->

In [ ]:
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", "mu_zhang2020")

## DEV TIMELYMT POLICIES
Run P0/P1/P2 causally across every preregistered probability threshold on DEV.
<!-- Compatibility heading: ## DEV LEARNED ROLLOUT -->

In [ ]:
LEARNED = [
    f"learned_{variant}_{threshold:.2f}"
    for variant in ("P0", "P1", "P2")
    for threshold in (0.30, 0.40, 0.50, 0.60, 0.70)
]
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *LEARNED)

## DEV EVALUATION
Compute frozen SacreBLEU, chrF2, AL, LAAL, and supporting latency/commit statistics from complete DEV predictions.
<!-- Compatibility heading: ## DEV EVALUATE -->

In [ ]:
BASELINES = FIXED + STYLE + ["local_agreement_la2", "mu_zhang2020"]
ALL_STRATEGIES = BASELINES + LEARNED
cli("evaluate", "--split", "dev", "--strategies", *ALL_STRATEGIES)
metrics = json.loads(require_file("outputs/experiments/research-mvp/metrics/dev/all.json").read_text(encoding="utf-8"))
if set(metrics) != set(ALL_STRATEGIES) or any(row.get("artifact_status") != "full" for row in metrics.values()):
    raise RuntimeError("DEV metrics are incomplete or not full")

## DEV SELECTION
Apply the frozen deterministic TimelyMT-only P0/P1/P2 selection rule. MU and LA-2 remain comparison baselines and cannot alter selection.
<!-- Compatibility heading: ## DEV SELECT -->

In [ ]:
cli("select")
selection = json.loads(require_file("outputs/experiments/research-mvp/dev-selection.json").read_text(encoding="utf-8"))
if not selection.get("selected_strategy", "").startswith("learned_P"):
    raise RuntimeError(f"Invalid TimelyMT DEV selection: {selection}")

## FREEZE FINAL EVALUATION CONFIG
Create the immutable evaluation configuration only after all full supervision, checkpoints, DEV metrics, and DEV selection pass their gates.
<!-- Compatibility heading: ## FREEZE -->

In [ ]:
import hashlib

cli("freeze")
frozen_path = require_file("outputs/experiments/research-mvp/frozen-eval-config.json")
frozen = json.loads(frozen_path.read_text(encoding="utf-8"))
required_frozen_keys = {
    "dataset_checksum", "split_checksum", "translator",
    "trained_checkpoint_hashes", "selected_learned_variant",
    "selected_learned_threshold", "baseline_config",
    "evaluation_metric_config",
}
if not required_frozen_keys.issubset(frozen):
    raise RuntimeError(f"Frozen config missing keys: {sorted(required_frozen_keys - set(frozen))}")
if frozen["dataset_checksum"] != DATASET_CHECKSUM or frozen["split_checksum"] != SPLIT_CHECKSUM:
    raise RuntimeError("Frozen config has unexpected Dataset v1 identity")
print(f"Frozen config: {frozen_path}")
print(f"SHA-256: {hashlib.sha256(frozen_path.read_bytes()).hexdigest()}")

## EXPORT ARTIFACTS
Package generated TRAIN/DEV supervision, trained policy checkpoints, DEV predictions/metrics/selection, frozen config, and compact provenance. Canonical Dataset v1 and all model/translator caches are excluded.

In [ ]:
import platform
import tarfile

experiment_dir = repo / "outputs/experiments/research-mvp"
provenance_path = experiment_dir / "kaggle-run-provenance.json"
provenance_path.write_text(json.dumps({
    "repository_url": REPO_URL,
    "repository_branch": REPO_BRANCH,
    "repository_commit": commit,
    "dataset_checksum": DATASET_CHECKSUM,
    "split_checksum": SPLIT_CHECKSUM,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "gpu_name": torch.cuda.get_device_name(0),
}, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")

EXPORT = Path("/kaggle/working/timelymt-research-mvp-artifacts.tar.gz")
ARTIFACT_DIRS = [
    Path("data/policy/pseudo_labels/train"),
    Path("data/policy/pseudo_labels/dev"),
    Path("data/policy/mu_zhang2020/train"),
    Path("data/policy/mu_zhang2020/dev"),
    Path("outputs/experiments/research-mvp"),
]
ARTIFACT_FILES = [
    *(Path("checkpoints/policy") / name for name in (
        "P0.joblib", "P0.metadata.json", "P1.joblib", "P1.metadata.json",
        "P2.joblib", "P2.metadata.json",
        "mu_zhang2020.joblib", "mu_zhang2020.metadata.json",
    )),
    Path("data/manifests/streaming-dataset.json"),
    Path("data/manifests/timelymt-streaming-dataset-v1.json"),
    Path("data/splits/experimental.json"),
    Path("configs/translator/envit5.json"),
    Path("configs/experiments/research-mvp.json"),
]
for relative in ARTIFACT_DIRS:
    if not (repo / relative).is_dir():
        raise RuntimeError(f"Missing required artifact directory: {repo / relative}")
for relative in ARTIFACT_FILES:
    require_file(relative)

def archive_filter(info):
    parts = Path(info.name).parts
    forbidden = {".git", "__pycache__", "cache"}
    return None if forbidden.intersection(parts) else info

with tarfile.open(EXPORT, "w:gz") as archive:
    for relative in ARTIFACT_DIRS:
        archive.add(repo / relative, arcname=relative.as_posix(), filter=archive_filter)
    for relative in ARTIFACT_FILES:
        archive.add(repo / relative, arcname=relative.as_posix(), filter=archive_filter)

with tarfile.open(EXPORT, "r:gz") as archive:
    top_levels = sorted({Path(member.name).parts[0] for member in archive.getmembers() if member.name})
print(f"Archive: {EXPORT}")
print(f"Archive size: {EXPORT.stat().st_size} bytes ({EXPORT.stat().st_size / 1024**2:.2f} MiB)")
print(f"Top-level archived directories: {top_levels}")

# STOP BEFORE TEST
TRAIN is complete, DEV selection is complete, and the experiment configuration is frozen. Execute the held-out evaluation separately using this frozen configuration. Do not use held-out data to tune thresholds, features, policies, baselines, or any other research choice. Kaggle exposes `/kaggle/working/timelymt-research-mvp-artifacts.tar.gz` as the downloadable notebook output.